# PyVista-ITK Quick Start

This notebook demonstrates the basic functionality of pyvista-itk.

In [ ]:
# Import required libraries
import numpy as np
import itk
import pyvista as pv
from pyvista_itk import itk_image_to_pyvista_grid, pyvista_grid_to_itk_image, data

# Set up notebook plotting
pv.set_jupyter_backend('static')

## 1. Create and Convert Synthetic Data

In [ ]:
# Create a synthetic gradient image
image = data.create_synthetic_image(
    size=(50, 50, 30),
    spacing=(1.0, 1.0, 2.0),
    pattern="gradient"
)

print(f"Created ITK image with size: {list(image.GetLargestPossibleRegion().GetSize())}")
print(f"Spacing: {list(image.GetSpacing())}")
print(f"Origin: {list(image.GetOrigin())}")

In [ ]:
# Convert to PyVista
grid = itk_image_to_pyvista_grid(image)

print(f"PyVista grid dimensions: {grid.dimensions}")
print(f"Number of points: {grid.n_points}")
print(f"Data range: {grid.get_data_range()}")

In [ ]:
# Visualize
plotter = pv.Plotter()
plotter.add_volume(grid, cmap="viridis", opacity="linear")
plotter.add_axes()
plotter.show()

## 2. Download and Visualize Example Data

In [ ]:
# List available examples
print("Available example datasets:")
for name, description in data.examples().items():
    print(f"  - {name}: {description}")

In [ ]:
# Download and load brain MRI
brain_image = data.load_example_brain_mri()
brain_grid = itk_image_to_pyvista_grid(brain_image)

# Visualize with slicing
plotter = pv.Plotter()
plotter.add_volume(brain_grid, cmap="gray", opacity="sigmoid")
plotter.add_axes()
plotter.show()

## 3. Round-trip Conversion

In [ ]:
# Create a sphere
sphere_image = data.create_synthetic_image(pattern="sphere")

# Convert ITK -> PyVista -> ITK
pv_grid = itk_image_to_pyvista_grid(sphere_image)
back_to_itk = pyvista_grid_to_itk_image(pv_grid)

# Verify the conversion preserved the data
original_array = itk.GetArrayFromImage(sphere_image)
converted_array = itk.GetArrayFromImage(back_to_itk)

print(f"Arrays are equal: {np.allclose(original_array, converted_array)}")
print(f"Maximum difference: {np.max(np.abs(original_array - converted_array))}")

## 4. Combine ITK Processing with PyVista Visualization

In [ ]:
# Create a checkerboard pattern
checkerboard = data.create_synthetic_image(
    size=(60, 60, 30),
    pattern="checkerboard"
)

# Apply ITK Gaussian smoothing
smoothed = itk.smooth_recursive_gaussian_image_filter(
    checkerboard,
    sigma=3.0
)

# Convert both to PyVista
original_grid = itk_image_to_pyvista_grid(checkerboard)
smoothed_grid = itk_image_to_pyvista_grid(smoothed)

# Visualize side by side
plotter = pv.Plotter(shape=(1, 2))

plotter.subplot(0, 0)
plotter.add_text("Original", position="upper_edge")
plotter.add_volume(original_grid, cmap="coolwarm")

plotter.subplot(0, 1)
plotter.add_text("Smoothed", position="upper_edge")
plotter.add_volume(smoothed_grid, cmap="coolwarm")

plotter.link_views()
plotter.show()

## 5. Advanced Visualization with PyVista

In [ ]:
# Create a sphere and threshold it
sphere_img = data.create_synthetic_image(
    size=(80, 80, 80),
    pattern="sphere"
)
sphere_grid = itk_image_to_pyvista_grid(sphere_img)

# Apply threshold to extract the sphere
thresholded = sphere_grid.threshold(0.5)

# Extract surface
surface = thresholded.extract_surface()

# Visualize with surface rendering
plotter = pv.Plotter()
plotter.add_mesh(surface, color="red", smooth_shading=True)
plotter.add_mesh(sphere_grid.outline(), color="black")
plotter.add_axes()
plotter.enable_anti_aliasing()
plotter.show()

## 6. Clean Up Cache

In [ ]:
# Optionally clear the download cache
# data.clear_cache()